In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import IGate, XGate, YGate, ZGate, HGate, SGate, TGate
from qiskit.quantum_info import Operator, Statevector

import linalg_toolkit as lt

np.set_printoptions(precision=4, suppress=True)

RESULTS = []

def check(label, numpy_value, qiskit_value, atol=1e-10):
    """Compare a NumPy result to a Qiskit result and record pass/fail."""
    ok = bool(np.allclose(numpy_value, qiskit_value, atol=atol))
    RESULTS.append((label, ok))
    print(("PASS  " if ok else "FAIL  ") + label)
    return ok

QK_GATES = {"I": IGate(), "X": XGate(), "Y": YGate(), "Z": ZGate(),
            "H": HGate(), "S": SGate(), "T": TGate()}

## 1. Inner products and orthonormality
`Statevector.inner(other)` computes <self|other>, which is the same convention as `lt.inner_product`.

In [ ]:
labels = {"0": lt.ket0, "1": lt.ket1, "+": lt.ket_plus, "-": lt.ket_minus,
          "r": lt.ket_r, "l": lt.ket_l}

for a, b in [("0", "1"), ("+", "-"), ("r", "l"), ("0", "+"), ("0", "r"), ("r", "1")]:
    np_val = lt.inner_product(labels[a], labels[b])
    qk_val = Statevector.from_label(a).inner(Statevector.from_label(b))
    print(f"<{a}|{b}> = {np_val:.4f}")
    check(f"inner product <{a}|{b}>", np_val, qk_val)

print()
print("{|0>,|1>} orthonormal:", lt.is_orthonormal([lt.ket0, lt.ket1]))
print("{|+>,|->} orthonormal:", lt.is_orthonormal([lt.ket_plus, lt.ket_minus]))
print("{|0>,|+>} orthonormal:", lt.is_orthonormal([lt.ket0, lt.ket_plus]), "  (deliberately non-orthogonal, should be False)")
assert lt.is_orthonormal([lt.ket0, lt.ket1]) and lt.is_orthonormal([lt.ket_plus, lt.ket_minus])
assert not lt.is_orthonormal([lt.ket0, lt.ket_plus])

## 2. Outer products and projectors
Each projector |psi><psi| should satisfy P^2 = P and P^dagger = P. Qiskit's `DensityMatrix` of a pure state is the same object, so we compare against that.

In [ ]:
from qiskit.quantum_info import DensityMatrix

for name, ket in labels.items():
    P = lt.projector(ket)
    assert lt.is_projector(P), name
    check(f"projector |{name}><{name}| equals Qiskit DensityMatrix", P,
          DensityMatrix(Statevector.from_label(name)).data)

print("|+><+| =")
print(lt.projector(lt.ket_plus))

# Measurement probability <psi|P|psi> vs squared amplitudes
psi = lt.normalize([1, 2 - 1j])
p0 = lt.measurement_probability(psi, lt.projector(lt.ket0))
p1 = lt.measurement_probability(psi, lt.projector(lt.ket1))
print(f"\npsi = {psi}")
print(f"<psi|P0|psi> = {p0:.4f}, |amp0|^2 = {abs(psi[0])**2:.4f}")
print(f"<psi|P1|psi> = {p1:.4f}, |amp1|^2 = {abs(psi[1])**2:.4f}")
qk_probs = Statevector(psi).probabilities()
check("measurement probabilities <psi|P|psi> vs Statevector.probabilities()", [p0, p1], qk_probs)

In [ ]:
for name, U in lt.GATES.items():
    check(f"matrix for {name}", U, Operator(QK_GATES[name]).data)
    assert lt.is_unitary(U)
    check(f"{name} is unitary (NumPy vs Operator.is_unitary)", lt.is_unitary(U), Operator(QK_GATES[name]).is_unitary())

# Non-unitary test matrices must fail in both
bad = np.array([[1, 1], [0, 1]], dtype=complex)
print("\nnon-unitary matrix [[1,1],[0,1]]  NumPy:", lt.is_unitary(bad), " Qiskit:", Operator(bad).is_unitary())
check("non-unitary matrix rejected by both", lt.is_unitary(bad), Operator(bad).is_unitary())

## 4. Hand-calculated matrix-vector products
Each "hand" value is the result worked out on paper; it is typed in directly, independent of the toolkit.

| Calculation | Paper result |
|---|---|
| H\|0> | (1/sqrt2)(1, 1) |
| X H\|0> | (1/sqrt2)(1, 1)  (since \|+> is an X eigenstate) |
| S H\|0> | (1/sqrt2)(1, i) |
| T H S \|+> | (1/sqrt2)(e^{i pi/4}, 1) |

For the last one: S\|+> = (1, i)/sqrt2; H of that = ((1+i)/2, (1-i)/2); T multiplies the second entry by e^{i pi/4}, and (1-i) e^{i pi/4} = sqrt2, giving ((1+i)/2, sqrt2/2) = (e^{i pi/4}, 1)/sqrt2.

In [ ]:
s2 = np.sqrt(2)
hand = {
    "H|0>":      np.array([1, 1]) / s2,
    "XH|0>":     np.array([1, 1]) / s2,
    "SH|0>":     np.array([1, 1j]) / s2,
    "THS|+>":    np.array([np.exp(1j * np.pi / 4), 1]) / s2,
}

# NumPy: one explicit step per gate
step = lt.ket0
h0 = lt.H @ step
xh0 = lt.X @ h0
sh0 = lt.S @ h0
ths_plus = lt.T @ (lt.H @ (lt.S @ lt.ket_plus))
numpy_vals = {"H|0>": h0, "XH|0>": xh0, "SH|0>": sh0, "THS|+>": ths_plus}

# Qiskit: Statevector.evolve applies gates in the order given
zero, plus = Statevector.from_label("0"), Statevector.from_label("+")
qiskit_vals = {
    "H|0>":   zero.evolve(Operator(QK_GATES["H"])).data,
    "XH|0>":  zero.evolve(Operator(QK_GATES["H"])).evolve(Operator(QK_GATES["X"])).data,
    "SH|0>":  zero.evolve(Operator(QK_GATES["H"])).evolve(Operator(QK_GATES["S"])).data,
    "THS|+>": plus.evolve(Operator(QK_GATES["S"])).evolve(Operator(QK_GATES["H"])).evolve(Operator(QK_GATES["T"])).data,
}

for key in hand:
    print(f"{key:8s} hand={hand[key]}  numpy={numpy_vals[key]}  qiskit={qiskit_vals[key]}")
    check(f"{key}: hand vs NumPy", hand[key], numpy_vals[key])
    check(f"{key}: NumPy vs Qiskit", numpy_vals[key], qiskit_vals[key])

## 5. Composition, identities, and operator ordering
The circuit below applies S, then H, then T. Its operator must equal the matrix product `T @ H @ S`, **not** `S @ H @ T`.

In [ ]:
qc = QuantumCircuit(1)
qc.s(0); qc.h(0); qc.t(0)          # circuit reads left to right: S, H, T
U_circuit = Operator(qc)

U_np = lt.T @ lt.H @ lt.S           # matrix order: rightmost acts first
U_wrong = lt.S @ lt.H @ lt.T

check("circuit S,H,T  ==  T @ H @ S", U_np, U_circuit.data)
print("circuit S,H,T  ==  S @ H @ T (wrong order)?", np.allclose(U_wrong, U_circuit.data), " (should be False)")
RESULTS.append(("wrong ordering S@H@T is NOT the circuit S,H,T", not np.allclose(U_wrong, U_circuit.data)))

# dot vs compose
OT, OH = Operator(QK_GATES["T"]), Operator(QK_GATES["H"])
check("Operator.dot:     T.dot(H)     == T @ H", lt.T @ lt.H, OT.dot(OH).data)
check("Operator.compose: T.compose(H) == H @ T", lt.H @ lt.T, OT.compose(OH).data)

In [ ]:
O = {k: Operator(v) for k, v in QK_GATES.items()}

identities = [
    ("HXH = Z",   lt.H @ lt.X @ lt.H,  lt.Z,        O["H"].dot(O["X"]).dot(O["H"])),
    ("HZH = X",   lt.H @ lt.Z @ lt.H,  lt.X,        O["H"].dot(O["Z"]).dot(O["H"])),
    ("HYH = -Y",  lt.H @ lt.Y @ lt.H, -lt.Y,        O["H"].dot(O["Y"]).dot(O["H"])),
    ("SS = Z",    lt.S @ lt.S,         lt.Z,        O["S"].dot(O["S"])),
    ("TT = S",    lt.T @ lt.T,         lt.S,        O["T"].dot(O["T"])),
    ("XY = iZ",   lt.X @ lt.Y,         1j * lt.Z,   O["X"].dot(O["Y"])),
    ("HH = I",    lt.H @ lt.H,         lt.I,        O["H"].dot(O["H"])),
]
for label, np_prod, expected, qk_prod in identities:
    check(f"{label}: NumPy product vs expected", np_prod, expected)
    check(f"{label}: Qiskit Operator product vs expected", qk_prod.data, expected)

# Same identity built as circuits
qc2 = QuantumCircuit(1); qc2.h(0); qc2.x(0); qc2.h(0)
check("circuit H,X,H equals Z", Operator(qc2).data, lt.Z)

## 6. Unitarity preserves inner products and norms
A numerical spot-check (the algebraic proof is in the Wong Ch. 3 notes): for every gate U and random states a, b, we need <Ua|Ub> = <a|b> and ||Ua|| = ||a||.

In [ ]:
rng = np.random.default_rng(42)
def rand_state():
    return lt.normalize(rng.normal(size=2) + 1j * rng.normal(size=2))

a, b = rand_state(), rand_state()
for name, U in lt.GATES.items():
    Ua = Statevector(a).evolve(Operator(QK_GATES[name])).data
    Ub = Statevector(b).evolve(Operator(QK_GATES[name])).data
    check(f"{name}: <Ua|Ub> == <a|b> (Qiskit)", lt.inner_product(Ua, Ub), lt.inner_product(a, b))
    check(f"{name}: ||Ua|| == 1 (Qiskit)", lt.norm(Ua), 1.0)

In [ ]:
failed = [label for label, ok in RESULTS if not ok]
print(f"{len(RESULTS) - len(failed)} / {len(RESULTS)} checks passed")
assert not failed, f"Failed checks: {failed}"